# Week 52, Day 4: Review IV: reward, decisions and cognition

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w052/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Problem 1: the prediction error moves to the cue

Simulate tabular TD(0) over 300 trials of 25 time steps. The cue arrives at t_cs and the reward (1) at t_rew. States before the cue cannot predict it, so their values stay at zero; states from the cue onwards learn. Record delta_t = r_t + gamma V[t + 1] - V[t] at every step. Then run one more trial with the reward omitted, without learning.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
T, t_cs, t_rew, alpha, gamma, n_trials = 25, 5, 15, 0.1, 0.98, 300
V = np.zeros(T + 1)
deltas = np.zeros((n_trials, T))

def run_trial(V, reward=1.0, learn=True):
    d = np.zeros(T)
    for t in range(T):
        r = reward if t == t_rew else 0.0
        d[t] = ...
        if learn and t >= t_cs:
            V[t] += alpha * d[t]
    return d

for k in range(n_trials):
    deltas[k] = run_trial(V)
omit = run_trial(V, reward=0.0, learn=False)
for k in [0, 10, 50, 299]:
    print(f"trial {k:3d}: error at cue {deltas[k, t_cs - 1]:.2f}, at reward {deltas[k, t_rew]:.2f}")
print(f"reward omitted after training: error at the reward time {omit[t_rew]:.2f}")
plt.imshow(deltas[:100], aspect="auto", cmap="RdBu_r", vmin=-1, vmax=1); plt.xlabel("time step"); plt.ylabel("trial"); plt.colorbar(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "deltas holds one row of TD errors per trial"
    assert deltas.shape == (n_trials, T)
def test_early():
    "on the first trial the error is at the reward, not the cue"
    assert deltas[0, t_rew] > 0.9 and abs(deltas[0, t_cs - 1]) < 1e-9
def test_late():
    "after training the error is at the cue and has nearly vanished at the reward"
    assert deltas[-1, t_cs - 1] > 0.7 and abs(deltas[-1, t_rew]) < 0.05
def test_omit():
    "omitting reward after training gives a negative error at the reward time"
    assert omit[t_rew] < -0.7

test_shape()
test_early()
test_late()
test_omit()
print("All checks passed")

### Exercise: Problem 2: the bound that maximises reward rate

Write accuracy(a) and mean_dt(a) from the formulas with drift v and noise variance s2. Check them by simulating 2,000 decisions at a = 1 (Euler steps of 1 ms). Then compute reward rate, accuracy over (decision time + t0 + iti), on a grid of bounds, and find the best bound for iti = 2 s and for iti = 8 s.

In [ ]:
v, s2, t0 = 1.0, 1.0, 0.3
grid = np.linspace(0.05, 3, 120)

def accuracy(a):
    return ...

def mean_dt(a):
    return ...

g = np.random.default_rng(45)
x = np.zeros(2000); t = np.zeros(2000); done = np.zeros(2000, bool); hit_up = np.zeros(2000, bool); dt = 0.001; k = 0
while not done.all() and k < 20000:
    live = ~done
    x[live] += v * dt + np.sqrt(s2 * dt) * g.normal(size=live.sum()); t[live] += dt
    up, down = live & (x >= 1.0), live & (x <= -1.0)
    hit_up |= up; done |= up | down; k += 1
sim_acc, sim_dt = hit_up.mean(), t.mean()

def reward_rate(a, iti):
    return ...

rate = reward_rate(grid, 2.0)
a_best = grid[np.argmax(rate)]
a_best_long = grid[np.argmax(reward_rate(grid, 8.0))]
print(f"a = 1: formula accuracy {accuracy(1.0):.3f}, simulated {sim_acc:.3f}; formula DT {mean_dt(1.0):.3f} s, simulated {sim_dt:.3f} s")
print(f"best bound with a 2 s wait: {a_best:.2f} (accuracy {accuracy(a_best):.2f}, DT {mean_dt(a_best):.2f} s); with an 8 s wait: {a_best_long:.2f}")

In [ ]:
import numpy as np
def test_formulas():
    "accuracy and mean decision time follow the DDM formulas"
    assert abs(accuracy(1.0) - 1 / (1 + np.exp(-2 * v * 1.0 / s2))) < 1e-12 and abs(mean_dt(1.0) - (1.0 / v) * np.tanh(v / s2)) < 1e-12
def test_sim():
    "the simulation at a = 1 agrees with the formulas"
    assert abs(sim_acc - accuracy(1.0)) < 0.03 and abs(sim_dt - mean_dt(1.0)) < 0.05
def test_opt():
    "a_best maximises reward rate on the grid and is an intermediate bound"
    assert a_best == grid[np.argmax(rate)] and grid[0] < a_best < grid[-1]
def test_iti():
    "a longer wait between trials raises the best bound"
    assert a_best_long > a_best

test_formulas()
test_sim()
test_opt()
test_iti()
print("All checks passed")

### Exercise: Problem 3: the width of a slow-speed prior

An observer with a Gaussian prior on speed centred at zero, of width sigma_p, sees a speed v with Gaussian likelihood of width sigma_l and reports the posterior mean. Write perceived(v, sigma_l, sigma_p). The noisy reports at five contrasts are simulated for you; fit sigma_p by minimising the squared error between the reports and the model.

In [ ]:
from scipy.optimize import minimize_scalar
g = np.random.default_rng(48)
like_sd = np.array([3.0, 2.0, 1.2, 0.7, 0.4])        # likelihood width falls as contrast rises
v_true, sigma_p_true = 4.0, 2.0

def perceived(v, sigma_l, sigma_p):
    return ...

reports = np.array([perceived(v_true, s, sigma_p_true) for s in like_sd])[:, None] + g.normal(0, 0.15, (5, 20))
loss = lambda sp: np.sum((reports - np.array([perceived(v_true, s, sp) for s in like_sd])[:, None]) ** 2)
sigma_p_hat = ...
for s, rep in zip(like_sd, reports.mean(1)):
    print(f"likelihood SD {s:.1f}: mean report {rep:.2f} (true speed {v_true})")
print(f"fitted prior width {sigma_p_hat:.2f} (true {sigma_p_true})")

In [ ]:
import numpy as np
def test_model():
    "perceived gives the posterior mean speed for a Gaussian prior centred on zero"
    assert abs(perceived(4.0, 0.5, 2.0) - 4.0 * 4.0 / (4.0 + 0.25)) < 1e-12
def test_fit():
    "sigma_p_hat recovers the true prior width within 15%"
    assert abs(sigma_p_hat - 2.0) / 2.0 < 0.15
def test_order():
    "low-contrast stimuli are perceived as slower than high-contrast ones"
    assert perceived(4.0, like_sd[0], sigma_p_hat) < perceived(4.0, like_sd[-1], sigma_p_hat)

test_model()
test_fit()
test_order()
print("All checks passed")